This is a starter notebook for an updated module 5 of ML Zoomcamp

The code is based on the modules 3 and 4. We use the same dataset: [telco customer churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

In [11]:
# IMPORT PACKAGES
import pandas as pd
import numpy as np
import sklearn

In [12]:
# CHECK PACKAGE VERSIONS
print(f'pandas=={pd.__version__}')
print(f'numpy=={np.__version__}')
print(f'sklearn=={sklearn.__version__}')

pandas==2.3.1
numpy==2.0.1
sklearn==1.7.1


In [13]:
# IMPORT feature_extraction AND LogisticRegression MODEL FUNCTIONS
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [14]:
# PART 1 - PREPARE DATA
#  1.0 GET DATA
data_url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(data_url)

#  2.0 NORMALIZE COLUMN NAMES
df.columns = df.columns.str.lower().str.replace(' ', '_')

#  3.0 CREATE RAW LIST OF CATEGORICAL COLUMNS
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

#  4.0 FILL MISSING NUMERIC VALUES
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

#  5.0 ONE-HOT ENCODE df.churn
df.churn = (df.churn == 'yes').astype(int)

In [15]:
#  6.0 CREATE TARGET LABELS VECTOR
y_train = df.churn

#  7.0 CREATE LIST OF NUMERICAL COLUMNS
numerical = ['tenure', 'monthlycharges', 'totalcharges']

#  8.0 CREATE SMALLER LIST OF CATEGORICAL COLUMNS BY INTENTIONALLY DROPPING COLUMNS
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

In [16]:
# PART 2 - CREATE THE MODEL
dv = DictVectorizer()

train_dict = df[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

model = LogisticRegression(solver='liblinear')
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'liblinear'
,max_iter,100
,multi_class,'deprecated'


In [17]:
# PART 3 - SPOT TESTING
#   1.0 CREATE A DATAPOINT TO USE WHEN SPOT TESTING A MODEL PREDICTION
datapoint = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}

In [18]:
#  2.0 - APPLY TRAINING WEIGHTS TO THE TEST DATAPOINT
X = dv.transform(datapoint)

#  3.0 - SPOT CHECK A MODEL PREDICTION
model.predict_proba(X)[0, 1]

np.float64(0.6638150320941973)

In [19]:
# PART 4 - USE PICKLE TO CONVERT VECTORIZER AND MODEL TO A BYTE STREAM FILE
#  NOTE: pickle CONVERTS PYTHON OBJECTS INTO BYTE STREAMS
import pickle

In [20]:
# USE pickle TO WRITE BYTES FROM THE DICTIONARY VECTORIZER INSTANCE 
#   AND THE MODEL TO A FILE NAMED 'model.bin'
with open('model.bin', 'wb') as f_out:
    pickle.dump((dv, model), f_out)

In [21]:
# USE pickle TO READ BYTES FROM model.bin INTO THE DICTIONARY VECTORIZER 
#   INSTANCE AND THE MODEL
with open('model.bin', 'rb') as f_in:
    (dv, model) = pickle.load(f_in)

In [22]:
# PART 5 - CREATE A PIPELINE
from sklearn.pipeline import make_pipeline

In [23]:
#   1.0 - CREATE A pipeline OBJECT THAT CREATES BOTH THE DictVectorizer AND THE model
pipeline = make_pipeline(
    DictVectorizer(),
    LogisticRegression(solver='liblinear')
)

In [24]:
#   NOTE: When you call pipeline.fit(X, y), the vectorizer learns the feature layout 
#         and transforms the dictionaries, then logistic regression trains on the 
#         resulting vectors. 
pipeline.fit(train_dict, y_train)

,steps,"[('dictvectorizer', ...), ('logisticregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,dtype,<class 'numpy.float64'>
,separator,'='
,sparse,True
,sort,True
,penalty,'l2'
,dual,False
,tol,0.0001


In [25]:
#   2.0 - MAKE A PREDICTION
pipeline.predict_proba(datapoint)[0, 1]

np.float64(0.6638150320941973)

In [28]:
# PART 5 - fastapi SETUP
import requests

In [27]:
url = 'http://localhost:9696/predict'

customer = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}

response = requests.post(url, json=customer)

ConnectionError: HTTPConnectionPool(host='localhost', port=9696): Max retries exceeded with url: /predict (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x12cd56e90>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [ ]:
predictions = response.json()

In [ ]:
if predictions['churn']:
    print('accept loan application')
else:
    print('reject loan application')

In [ ]:
for n in numerical:
    print(df[n].describe())
    print()

for c in categorical:
    print(df[c].value_counts())
    print()